# SNA Experiment 1 – Coding Social Networks
**Runtime:** Runtime → Change runtime type → **R**. Then Runtime → Run all.

Congress members = **vertices (nodes)**. Two members are joined by an **edge** if they sit on the same committee.
The edge **weight** = how many committees they share.

## Setup: install and load packages (install takes ~2–3 min the first time in a session)

In [ ]:
if (!requireNamespace("igraph", quietly = TRUE)) install.packages("igraph")
if (!requireNamespace("ggraph", quietly = TRUE)) install.packages("ggraph")

library(dplyr)
library(igraph)
library(ggraph)

## Load the data

In [ ]:
load(url("https://dssoc.github.io/datasets/congress.RData"))      # congress, congress_contact
load(url("https://dssoc.github.io/datasets/committees.RData"))    # committees, subcommittees, committee_memberships

head(congress)
head(committee_memberships)

## Build `common_committees` (one row per pair of members + how many committees they share)
1. Keep only memberships of **full committees** (not subcommittees) and only members present in `congress`.
2. **Self-join** on `thomas_id`: every pair of members on the same committee becomes a row.
3. Keep each pair once (`x < y`), then count rows per pair = number of shared committees.

In [ ]:
memberships <- committee_memberships %>%
  filter(thomas_id %in% committees$thomas_id) %>%
  filter(bioguide_id %in% congress$bioguide_id) %>%
  select(thomas_id, bioguide_id) %>%
  distinct()

common_committees <- memberships %>%
  inner_join(memberships, by = "thomas_id", relationship = "many-to-many") %>%
  filter(bioguide_id.x < bioguide_id.y) %>%
  count(bioguide_id.x, bioguide_id.y, name = "n_committees")

head(common_committees)

## (a) igraph graph from `common_committees` – number of vertices and edges

In [ ]:
g <- graph_from_data_frame(common_committees, directed = FALSE, vertices = congress)

cat("Vertices:", vcount(g), "\n")
cat("Edges:", ecount(g), "\n")

## (b) Edges only between members on **at least 3** committees together

In [ ]:
common_3 <- common_committees %>%
  filter(n_committees >= 3)

g3 <- graph_from_data_frame(common_3, directed = FALSE, vertices = congress)

cat("Edges (3+ shared committees):", ecount(g3), "\n")

# who is most connected? degree = number of edges of a vertex
top <- sort(degree(g3), decreasing = TRUE)[1:5]
data.frame(member = V(g3)[names(top)]$full_name, degree = top)

## (c) Visualise the previous network with ggraph (isolated members removed so the plot is readable)

In [ ]:
g3_plot <- delete_vertices(g3, degree(g3) == 0)
cat("Members shown:", vcount(g3_plot), "\n")

ggraph(g3_plot, layout = "fr") +
  geom_edge_link(aes(width = n_committees), alpha = 0.3) +
  geom_node_point(aes(color = party), size = 3) +
  scale_edge_width(range = c(0.5, 2)) +
  scale_color_manual(values = c(Democrat = "blue", Independent = "darkgreen", Republican = "red")) +
  ggtitle("Congress members sharing at least 3 committees")

## (d) Network with **only senators** (subgraph of the full network)

In [ ]:
senator_ids <- congress %>%
  filter(type == "sen") %>%
  pull(bioguide_id)

g_sen <- induced_subgraph(g, senator_ids)

cat("Senators (vertices):", vcount(g_sen), "\n")
cat("Edges between senators:", ecount(g_sen), "\n")

ggraph(g_sen, layout = "fr") +
  geom_edge_link(alpha = 0.1) +
  geom_node_point(aes(color = party), size = 3) +
  scale_color_manual(values = c(Democrat = "blue", Independent = "darkgreen", Republican = "red")) +
  ggtitle("Senate committee network")